<a href="https://colab.research.google.com/github/beyzaturku/2209/blob/main/SRCNN/SR_dataset_split.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import cv2
import numpy as np
from sklearn.model_selection import train_test_split
from google.colab import drive

In [2]:
drive.mount('/content/drive')

Mounted at /content/drive


* 4x downsampling ve bicubic interpolasyonu ile LR görüntü elde etme

In [14]:
# Veri seti yolu
image_folder = "/content/drive/MyDrive/srcnn_dataset/dataset/train"  # Mevcut görüntülerin bulunduğu klasör
hr_train_folder = "/content/drive/MyDrive/srcnn_dataset/SRCNN_Dataset/HR/train"
hr_test_folder = "/content/drive/MyDrive/srcnn_dataset/SRCNN_Dataset/HR/test"
lr_train_folder = "/content/drive/MyDrive/srcnn_dataset/SRCNN_Dataset/LR/train"
lr_test_folder = "/content/drive/MyDrive/srcnn_dataset/SRCNN_Dataset/LR/test"

# Klasörleri oluştur
os.makedirs(hr_train_folder, exist_ok=True)
os.makedirs(lr_train_folder, exist_ok=True)
os.makedirs(hr_test_folder, exist_ok=True)
os.makedirs(lr_test_folder, exist_ok=True)

# Downsampling oranı 4x seçildi
scale_factor = 4

# Görüntüleri al
image_files = [f for f in os.listdir(image_folder) if f.endswith('.jpg') or f.endswith('.png')]

# Görüntüleri eğitim ve test olarak ayır
train_images, test_images = train_test_split(image_files, test_size=0.2, random_state=42)

# Görüntüleri 4x oranında küçültmek ve kaydetmek için:
def downsample_and_save(image_list, hr_folder, lr_folder):
    for image_name in image_list:
        #Görüntüyü yükle
        img_path = os.path.join(image_folder, image_name)
        img = cv2.imread(img_path)

        # Görüntüyü normalleştir bölme işlemi, piksel değerlerinin [0, 1] aralığında olması için)
        img = img.astype(np.float32) / 255.0

        # HR görüntüyü kaydet
        hr_path = os.path.join(hr_folder, image_name)
        cv2.imwrite(hr_path, img * 255)

        #4x oranında küçültülerek ve bicubic interpolasyon ile LR görüntü oluşturulur
        lr_img = cv2.resize(img, (img.shape[1] // scale_factor, img.shape[0] // scale_factor), interpolation=cv2.INTER_CUBIC)

        #LR görüntüyü kaydet
        lr_path = os.path.join(lr_folder, image_name)
        cv2.imwrite(lr_path, lr_img * 255)

# Eğitim görüntüleri için metodu uygula
print("Eğitim görüntüleri işleniyor:")
downsample_and_save(train_images, hr_train_folder, lr_train_folder)
print("Eğitim görüntüleri kaydedildi.")

print("Test görüntüleri işleniyor:")
downsample_and_save(test_images, hr_test_folder, lr_test_folder)
print("Test görüntüleri kaydedildi.")

Eğitim görüntüleri işleniyor:
Eğitim görüntüleri kaydedildi.
Test görüntüleri işleniyor:
Test görüntüleri kaydedildi.


* LR görüntüyü 4x upscale ederek HR görüntü elde etme

In [19]:
# LR klasörleri ve HR klasörleri
lr_train_folder = "/content/drive/MyDrive/srcnn_dataset/SRCNN_Dataset/LR/train"
lr_test_folder = "/content/drive/MyDrive/srcnn_dataset/SRCNN_Dataset/LR/test"

lr_upscale_train_folder = "/content/drive/MyDrive/srcnn_dataset/SRCNN_Dataset/LR_upscale/train"
lr_upscale_test_folder = "/content/drive/MyDrive/srcnn_dataset/SRCNN_Dataset/LR_upscale/test"

os.makedirs(lr_upscale_train_folder, exist_ok=True)
os.makedirs(lr_upscale_test_folder, exist_ok=True)

#Görüntülerin listesi
lr_train_images = [f for f in os.listdir(lr_train_folder) if f.endswith('.jpg') or f.endswith('.png')]
lr_test_images = [f for f in os.listdir(lr_test_folder) if f.endswith('.jpg') or f.endswith('.png')]

#upscale oranı = 4
scale_factor = 4

# LR görüntüleri 4x bicubic interpolasyon ile büyüt
def upscale_and_save(lr_image_list, lr_folder, hr_folder):
  for image_name in lr_image_list:
    #Görüntüyü yükle
    img_path = os.path.join(lr_folder, image_name)
    img = cv2.imread(img_path)

    #Görüntüyü normalleştir
    img = img.astype(np.float32) / 255.0

    #Orijinal boyutlarına 4x büyütme
    h, w = img.shape[:2]
    hr_img = cv2.resize(img, (w * scale_factor, h * scale_factor), interpolation=cv2.INTER_CUBIC)

    # HR görüntüyü kaydet
    hr_path = os.path.join(hr_folder, image_name)
    cv2.imwrite(hr_path, hr_img * 255)

print("Upscale edilmiş görüntüler oluşturuluyor:")
upscale_and_save(lr_train_images, lr_train_folder, lr_upscale_train_folder)
upscale_and_save(lr_test_images, lr_test_folder, lr_upscale_test_folder)
print("Upscale edilmiş görüntüler oluşturuldu.")

Upscale edilmiş görüntüler oluşturuluyor:
Upscale edilmiş görüntüler oluşturuldu.


# Veri seti hazırlandı.